# 🌐 Web Auth Signing을 사용하는 AgentCore Browser

## 개요

이 튜토리얼에서는 Amazon Bedrock AgentCore Browser tool에서 Web Bot Auth signing을 활성화하는 방법을 알아봅니다.

### 튜토리얼 세부 정보

| 항목                | 세부 정보                                                                        |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 대화형                                                                           |
| Agent 유형          | 단일                                                                             |
| Agentic Framework   | Strands                                                                          |
| LLM 모델            | Claude 4.5 Haiku                                                                 |
| 튜토리얼 구성 요소  | 브라우저 자동화, 브라우저 request의 *Web Bot Auth* signing                      |
| 튜토리얼 분야       |                                                                                  |
| 예제 난이도         | 중급                                                                             |
| 사용 SDK            | Amazon Bedrock AgentCore Python SDK, Strands Agents, Strands Agent Tools         |
 
### 튜토리얼 아키텍처

![Architecture](images/Architecture.png)

### 튜토리얼 주요 기능

* Strands Agent와 Browser tool을 headless 방식으로 사용
* strands_tools AgentCoreBrowser tool 사용
* 빠르고 효율적인 분석을 위한 Claude 4.5 Haiku 모델

## 사전 요구 사항

이 튜토리얼을 실행하려면 다음이 필요합니다.
* Python 3.10+
* 구성된 AWS 자격 증명
* Amazon Bedrock AgentCore SDK
* Strands Agent 및 strands-agents-tools package
* Amazon Bedrock의 Claude 4.5 Haiku 모델에 대한 액세스 권한

**구현**: 공식 `strands_tools.browser.AgentCoreBrowser`를 사용해 Agent에서 브라우저를 자동화합니다.

## 🔧 AgentCore Browser 구성

AgentCore Browser tool을 사용하기 전에 녹화 기능, network configuration, execution role 등의 설정이 포함된 사용자 지정 브라우저 구성을 선택적으로 생성할 수 있습니다. 이 섹션에서는 AWS SDK를 사용해 사용자 지정 브라우저 구성을 생성하는 방법을 설명합니다.

### 사용자 지정 브라우저 구성 생성

다음 코드는 녹화가 활성화되고 특정 execution role이 지정된 사용자 지정 AgentCore Browser를 생성하는 방법을 보여 줍니다.

#### 📚 Web Bot Auth Signing이 적용된 브라우저 구성 생성

**Browser Signing이란?**

Browser signing(`browserSigning.enabled = True`)은 AgentCore Browser가 모든 발신 HTTP request에 자동으로 서명하도록 구성합니다. 다음과 같은 경우에 필수적입니다.

- **인증된 API 액세스**: 보호된 API에 보내는 request 서명
- **AWS 서비스 통합**: AWS 자격 증명으로 request 자동 서명
- **보안 규정 준수**: 모든 브라우저 request의 인증 보장
- **Request 무결성**: Request header에 암호학적 서명 적용

활성화하면 브라우저는 다음 작업을 수행합니다.
1. 모든 HTTP/HTTPS request 가로채기
2. Request header에 암호학적 서명 추가
3. 인증 token 자동 포함
4. 여러 request에서 세션 무결성 유지


### 📦 라이브러리 import


In [ ]:
import boto3
import uuid
import os
import sys
from strands import Agent
from strands_tools.browser import AgentCoreBrowser

tutorials_path = os.path.abspath(os.path.join(os.getcwd(), "../../../"))
if tutorials_path not in sys.path:
    sys.path.insert(0, tutorials_path)

from utils import create_agentcore_role

cp_client = boto3.client("bedrock-agentcore-control", region_name="us-west-2")


accountId = boto3.client("sts").get_caller_identity()["Account"]
region = boto3.Session().region_name

print(f"Account ID: {accountId}")
print(f"Region: {region}")

### 🔧 설정 및 Role 생성


In [ ]:
## Execution role 생성
execution_role_arn = create_agentcore_role("web-bot-auth")["Role"]["Arn"]

print(f"\n✅ Role Created Successfully : {execution_role_arn}")

## 사용자 지정 구성으로 새 브라우저 instance 생성
response = cp_client.create_browser(
    name="web_bot_auth_browser_" + str(uuid.uuid4())[:6],
    description="Browser configured to sign web bot auth",
    networkConfiguration={"networkMode": "PUBLIC"},
    executionRoleArn=execution_role_arn,
    browserSigning={"enabled": True},
)

browserId = response["browserId"]
browserArn = response["browserArn"]
print("\n✅ Browser Created Successfully!")
print(f"   Browser ID: {browserId}")
print(f"   Browser ARN: {browserArn}")
print("\n🔐 Browser signing is ENABLED - all requests will be automatically signed")

### 🔧 AgentCoreBrowser Tool을 사용하는 Strands Agent 생성


In [ ]:
# AgentCoreBrowser로 Strands Agent 생성 및 구성
# 이전 단계에서 생성한 사용자 지정 browser ID로 공식 AgentCoreBrowser
# tool 초기화
agent_core_browser = AgentCoreBrowser(identifier=browserId, region="us-west-2")
agent_core_default_browser = AgentCoreBrowser(region="us-west-2")

# 동시 브라우저 작업을 방지하기 위해 SequentialToolExecutor import
from strands.tools.executors import SequentialToolExecutor

# Claude 4.5 Haiku 모델과 순차 tool 실행을 사용하는 SIGNED Agent 생성
strands_agent = Agent(
    tools=[agent_core_browser.browser],  # Signing이 활성화된 사용자 지정 브라우저 사용
    tool_executor=SequentialToolExecutor(),  # 동시 브라우저 작업 방지
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    system_prompt="""You are a website analyst with browser signing capabilities.
1. Use the browser tool to visit and interact with the website EFFICIENTLY
2. Focus on extracting key information QUICKLY and within 2-3 browser interactions.
3. Review browser requests for signatures related to Web Bot Auth's Signature and Signature-Agent http headers, 
   to verify if browser signing is configured.""",
)

# Claude 4.5 Haiku 모델과 순차 tool 실행을 사용하는 UNSIGNED Agent 생성
strands_agent_unsigned = Agent(
    tools=[agent_core_default_browser.browser],  # Signing이 없는 기본 브라우저 사용
    tool_executor=SequentialToolExecutor(),  # 동시 브라우저 작업 방지
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    system_prompt="""You are a website analyst with browser signing capabilities.
1. Use the browser tool to visit and interact with the website EFFICIENTLY
2. Focus on extracting key information QUICKLY and within 2-3 browser interactions.
3. Review browser requests for signatures related to Web Bot Auth's Signature and Signature-Agent http headers, 
   to verify if browser signing is configured.""",
)

In [ ]:
# 동시 실행 충돌이 없도록 순차 실행 async function 정의
async def analyze_website(agent, prompt):
    """도구를 순차적으로 실행하며 에이전트를 호출하는 비동기 래퍼입니다."""
    try:
        # SequentialToolExecutor로 브라우저 작업 충돌 방지
        result = await agent.invoke_async(prompt)
        return result
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        import traceback

        traceback.print_exc()
        return None

### 🚀 Browser Signing을 활성화해 분석 시작


In [ ]:
print("\n🚀 Validate browser signing against CloudFlare's crawltest site.")
print("=" * 100)

result_signed = await analyze_website(
    strands_agent,
    "Review the output and status code at https://crawltest.com/cdn-cgi/web-bot-auth and provide 3 to 4 concise key insights, based on https://developers.cloudflare.com/bots/reference/bot-verification/web-bot-auth/",
)

if result_signed:
    print("\n\n✅ Analysis completed, with Web Bot Auth browser signing enabled")
    print("-" * 100)
    print(result_signed)
    print("-" * 100)

### Browser signing 없이 실험 다시 실행

테스트가 유효한지 확인하기 위해 동일한 prompt를 browser signing 없이 구성된 Agent로 실행해 보겠습니다.

In [ ]:
print("\n🚀 Validate browser signing against CloudFlare's crawltest site - without Web Bot Auth browser signing.")
print("=" * 100)

result_unsigned = await analyze_website(
    strands_agent_unsigned,
    "Review the output and status code at https://crawltest.com/cdn-cgi/web-bot-auth and provide 3 to 4 concise key insights, based on https://developers.cloudflare.com/bots/reference/bot-verification/web-bot-auth/",
)

if result_unsigned:
    print("\n\n✅ Analysis completed - without Web Bot Auth browser signing")
    print("-" * 100)
    print(result_unsigned)
    print("-" * 100)

결과를 비교해 보겠습니다.

In [ ]:
# Strands Agent를 사용해 result_unsigned와 result_signed 비교
comparison_prompt = f"""
Please analyze and compare these two agent outputs side by side:

**Unsigned Agent Output:**
{result_unsigned}

**Signed Agent Output:**
{result_signed}

Please provide:
* A side-by-side comparison highlighting key differences
* Validation of the Signed Agent using Signatures correctly
* Validation of the Unsigned Agent NOT using Signatures
* Summary of which aspects differ most significantly

Format your response clearly with headers and bullet points for easy reading.
"""

# 결과 비교를 위해 Strands Agent 호출
comparison_agent = Agent(
    system_prompt="""You are an expert data analyst evaluating different AI agent outputs.""",
    callback_handler=None,
)

comparison_response = comparison_agent(comparison_prompt)

print("=== COMPARISON OF AGENT OUTPUTS ===")
print(comparison_response)

## 🎭 내부에서는 어떤 일이 일어났을까요?

이 Notebook을 **browser signing이 활성화된 상태**로 실행하면 다음 프로세스가 진행됩니다.

### 1. **브라우저 구성 생성**
```python
browserSigning={
    "enabled": True
}
```
이 설정은 브라우저에서 생성되는 모든 HTTP request에 자동으로 서명하도록 AgentCore Browser 서비스에 지시합니다.

### 2. **Agent 초기화**
Strands Agent는 다음 항목으로 초기화됩니다.
- Signing이 활성화된 브라우저 구성 identifier. *참고*: 기본 브라우저 identifier가 아닙니다.
- Claude 4.5 Haiku 모델
- Browser tool 통합

### 3. **Request Signing 흐름**
Agent가 웹 사이트를 탐색할 때 다음 흐름으로 진행됩니다.

```text
┌─────────────────┐    ┌───────────────────┐    ┌──────────────────────┐    ┌──────────────┐
│   Agent 요청    │ ──→│ AgentCore Browser │ ──→│ Request Header 서명  │ ──→│ 대상 웹 사이트 │
└─────────────────┘    └───────────────────┘    └──────────────────────┘    └──────────────┘
                                                          │
                                                          ▼
                                                Web Bot Auth header 추가:
                                                • Signature-Input header
                                                • Signature-Agent header
                                                • Signature header

```

## 🔒 보안 이점

Browser signing을 활성화하면 다음 이점이 있습니다.

✅ **자동 인증** - 수동 header 관리 불필요  
✅ **Request 무결성** - 암호학적 서명으로 변조 방지  
✅ **AWS 통합** - Agent가 Browser tool을 사용하도록 IAM 자격 증명을 원활하게 통합
✅ **세션 관리** - 각 세션에 고유한 브라우저 세션을 부여하고 session token을 안전하게 처리
✅ **감사 추적** - 서명된 모든 request를 log로 기록 가능  

## 🛠️ 문제 해결

### Error: "Browser session not found"
**해결 방법**: Browser ID가 만료되었을 수 있습니다. 브라우저 생성 셀(cell-3)을 다시 실행하세요.

### RuntimeError: "Leaving task does not match the current task"
**해결 방법**: 브라우저 작업이 동시에 실행될 때 이 asyncio 오류가 발생합니다. 이 Notebook은 브라우저 작업이 한 번에 하나씩 실행되도록 `SequentialToolExecutor()`를 사용해 문제를 방지합니다.

**발생 원인**: Agent가 AgentCore Browser tool에 여러 async 작업(get_html, get_text, screenshot 등)을 병렬로 요청하면 Strands가 작업을 동시에 실행해 asyncio task 충돌이 발생할 수 있습니다.

**해결책**: `SequentialToolExecutor()`를 사용해 모든 tool 호출이 순차적으로 실행되도록 하면 동일한 기능을 유지하면서 async task 충돌을 제거할 수 있습니다.


## 📚 추가 자료

Web Bot Auth와 AI Agent의 CAPTCHA를 줄이는 방법을 자세히 알아보려면 다음 자료를 참고하세요.

### AWS 블로그 게시물
- **[Reduce CAPTCHAs for AI agents browsing the web with Web Bot Auth (Preview) in Amazon Bedrock AgentCore Browser](https://aws.amazon.com/blogs/machine-learning/reduce-captchas-for-ai-agents-browsing-the-web-with-web-bot-auth-preview-in-amazon-bedrock-agentcore-browser/)** - Web Bot Auth 구현과 이점에 관한 종합 가이드

### 문서
- **[Cloudflare Web Bot Auth Documentation](https://developers.cloudflare.com/bots/reference/bot-verification/web-bot-auth/)** - 기술 사양 및 구현 세부 정보
- **[Amazon Bedrock AgentCore Browser Documentation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/browser-onboarding.html)** - AgentCore Browser 기능 전체 가이드

### 관련 주제
- **[HTTP Message Signatures (RFC 9421)](https://datatracker.ietf.org/doc/rfc9421/)** - Web Bot Auth에서 사용하는 기반 암호화 표준
- **[Web Bot Auth Architecture (IETF Draft)](https://datatracker.ietf.org/doc/html/draft-meunier-web-bot-auth-architecture)** - Web Bot Auth 아키텍처의 IETF draft 사양